# Sanity checks 

### _Volume Check ✅_


In [0]:
%sql
SELECT COUNT(*) AS total_rows 
FROM shopstream.raw.raw_events;

### _Completeness Check ✅_

In [0]:
%sql
SELECT 
    COUNT(*) AS total_rows,
    COUNT(event_time) AS non_null_event_time,
    COUNT(user_id) AS non_null_users,
    COUNT(category_code) AS non_null_categories,
    COUNT(*) - COUNT(category_code) AS missing_categories
FROM shopstream.raw.raw_events;

### _Category column anomoly check ✅_

In [0]:
%sql
SELECT 
    event_type, 
    COUNT(*) AS event_count
FROM shopstream.raw.raw_events
GROUP BY event_type
ORDER BY event_count DESC;

### _Price anomoly check ✅_

In [0]:
%sql
SELECT 
    MIN(CAST(price AS DECIMAL(10,2))) AS min_price,
    MAX(CAST(price AS DECIMAL(10,2))) AS max_price
FROM shopstream.raw.raw_events;

### _Ghost clicks check ❌_
_Duplicates were found and will be further cleaned with dbt_transpformations_

Why?
Possible answers:
- Double-Clicke
- Network Retry
- Frontend Bugs

In [0]:
%sql
WITH CTE AS (SELECT 
    user_id,
    event_time,
    product_id,
    event_type,
    COUNT(*) AS duplicate_count
FROM shopstream.raw.raw_events
GROUP BY 
    user_id,
    event_time,
    product_id,
    event_type
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC)

SELECT 
    COUNT(*) AS duplicate_count
FROM CTE